In [1]:
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
ROOT = Path("/content/drive/MyDrive/PathSnap")

print("Does PathSnap folder exist?", ROOT.exists())

if ROOT.exists():
    print("\nFiles and folders inside PathSnap:")
    for p in sorted(ROOT.iterdir()):
        print(p.name)
else:
    print("\nPathSnap folder not found.")
    print("\nAvailable folders in MyDrive:")
    for p in Path("/content/drive/MyDrive").iterdir():
        print(p.name)

Does PathSnap folder exist? False

PathSnap folder not found.

Available folders in MyDrive:
PES1UG24CS643_Vedaraj H R (6).pdf
PES1UG24CS643_Vedaraj H R (5).pdf
Classroom
PES1UG24CS643_Vedaraj H R_Assignment1.zip
PES1UG24CS643_Vedaraj H R (4).pdf
Colab Notebooks
PES1UG24CS643_Vedaraj H R (3).pdf
Digital dawn_3k.pdf
PES1UG24CS643_Vedaraj H R (2).pdf
Resume_Vedaraj H R (4).pdf
Resume_Vedaraj H R (3).pdf
Resume_Vedaraj H R (2).pdf
Resume_Vedaraj H R (1).pdf
Resume_Vedaraj H R.pdf
PES1UG24CS643_Vedaraj H R (1).pdf
PES1UG24CS643_Vedaraj H R_Orange_Assignment.zip
PES1UG24CS643_Vedaraj H R.pdf
Onlinelearningportal_Vedarajhr.zip
Copy of R1.pdf
Vedaraj H R - Lab-1-Part-2.gdoc
PES1UG24CS643_VedarajHR_LAB1_2b.pdf
PES1UG24CS643_VedarajHR_LAB2-1.pdf
PES1UG24CS643_VedarajHR (1).pdf
PES1UG24CS643_VedarajHR.pdf
PES1UG24CS643_VedarajHR_LAB3.pdf
DAA Unit4S.pdf
CN Unit1S.pdf
DAA Unit2S.pdf
DAA Unit3S.pdf
MPCA Unit3S.pdf
CN Unit3S.pdf
MPCA Unit2S.pdf
DAA Unit1S.pdf
CN Unit4S.pdf
LAA Unit3S.pdf
LAA Unit4S.

In [3]:
from pathlib import Path
import zipfile

ROOT = Path("/content/drive/MyDrive/PathSnap")

ZIP_PATH = ROOT / "Student folder.zip"

print("ZIP file exists:", ZIP_PATH.exists())

assert ZIP_PATH.exists(), (
    "ZIP file not found. Check that you uploaded it "
    "inside the PathSnap folder in Google Drive."
)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(ROOT)

print("\nExtraction completed successfully!")

DATASET = ROOT / "Student folder"

print("\nDataset folder exists:", DATASET.exists())

print("\nDataset contents:")
for item in sorted(DATASET.iterdir()):
    print(item.name)

ZIP file exists: True

Extraction completed successfully!

Dataset folder exists: True

Dataset contents:
student_test
train


In [4]:
from collections import Counter

TRAIN_DIR = DATASET / "train"
TEST_DIR = DATASET / "student_test"

train_trips = sorted(
    p for p in TRAIN_DIR.iterdir() if p.is_dir()
)

test_trips = sorted(
    p for p in TEST_DIR.iterdir() if p.is_dir()
)

print("Training trips:", len(train_trips))
print("Student testing trips:", len(test_trips))

print("\nChecking files in every trip...")

required_extensions = [
    ".track",
    ".route",
    ".nodes",
    ".arcs"
]

missing = []

for trip_dir in train_trips + test_trips:
    for ext in required_extensions:
        expected_file = trip_dir / (trip_dir.name + ext)

        if not expected_file.exists():
            missing.append(str(expected_file))

print("Missing files:", len(missing))

if missing:
    print("\nMissing file details:")
    for f in missing:
        print(f)
else:
    print("All required files are present!")

assert len(train_trips) == 80
assert len(test_trips) == 10
assert len(missing) == 0

print("\nDATASET VERIFICATION PASSED!")

Training trips: 80
Student testing trips: 10

Checking files in every trip...
Missing files: 0
All required files are present!

DATASET VERIFICATION PASSED!


In [5]:
%pip install -q "shapely>=2.0" pyproj

In [6]:
import numpy as np
import pandas as pd
import shapely

from shapely import STRtree
from pyproj import Transformer

KEY = ["trip_id", "obs_id"]

def process_trip(split, trip_dir, radius=50, max_candidates=8):

    trip_id = trip_dir.name
    base = trip_dir / trip_id

    # Read the four original dataset files
    track = np.loadtxt(str(base) + ".track")
    nodes = np.loadtxt(str(base) + ".nodes")
    arcs = np.loadtxt(str(base) + ".arcs", dtype=int)
    route = np.loadtxt(str(base) + ".route", dtype=int).reshape(-1)

    track = np.atleast_2d(track)
    nodes = np.atleast_2d(nodes)
    arcs = np.atleast_2d(arcs)

    # Verify the supplied reference route and graph
    assert route.min() >= 0
    assert route.max() < len(arcs)
    assert arcs.min() >= 0
    assert arcs.max() < len(nodes)

    # Use a suitable local metric projection
    lon0, lat0 = np.median(track[:, :2], axis=0)
    zone = int((lon0 + 180) // 6) + 1
    epsg = (32600 if lat0 >= 0 else 32700) + zone

    transformer = Transformer.from_crs(
        "EPSG:4326",
        f"EPSG:{epsg}",
        always_xy=True
    )

    # Project road-node coordinates
    nx, ny = transformer.transform(
        nodes[:, 0], nodes[:, 1]
    )

    # Each row in arcs is one directed candidate edge.
    edge_geoms = [
        shapely.LineString([
            (nx[u], ny[u]),
            (nx[v], ny[v])
        ])
        for u, v in arcs
    ]

    edge_geoms = np.asarray(edge_geoms, dtype=object)

    # Project GPS observations
    gx, gy = transformer.transform(
        track[:, 0], track[:, 1]
    )

    point_geoms = shapely.points(gx, gy)

    # Reference edge for each GPS observation:
    # nearest edge from the supplied reference route.
    route_geoms = edge_geoms[route]
    route_tree = STRtree(route_geoms)

    nearest_route_pos = route_tree.nearest(point_geoms)
    ref_edge_ids = route[nearest_route_pos]

    obs_ids = [
        str(int(round(x))) for x in track[:, 2]
    ]

    obs_part = pd.DataFrame({
        "trip_id": trip_id,
        "obs_id": obs_ids,
        "ref_edge_id": ref_edge_ids.astype(str),
        "split": split
    })

    # Generate candidate edges within 50 metres
    tree = STRtree(edge_geoms)

    pairs = tree.query(
        point_geoms,
        predicate="dwithin",
        distance=radius
    )

    point_idx, edge_idx = pairs

    if len(point_idx) == 0:
        cand_part = pd.DataFrame(columns=[
            "trip_id", "obs_id", "edge_id",
            "distance_m", "distance_rank"
        ])
    else:
        distances = shapely.distance(
            point_geoms[point_idx],
            edge_geoms[edge_idx]
        )

        cand_part = pd.DataFrame({
            "trip_id": trip_id,
            "obs_id": np.array(obs_ids)[point_idx],
            "edge_id": edge_idx.astype(str),
            "distance_m": distances
        })

        cand_part = cand_part.sort_values(
            ["trip_id", "obs_id", "distance_m", "edge_id"]
        )

        cand_part = cand_part.drop_duplicates(
            ["trip_id", "obs_id", "edge_id"]
        )

        cand_part["distance_rank"] = (
            cand_part.groupby(KEY).cumcount() + 1
        )

        cand_part = cand_part[
            cand_part["distance_rank"] <= max_candidates
        ].copy()

    return obs_part, cand_part


# Test on two training trips
sample_ids = ["00000001", "00000003"]

sample_obs = []
sample_candidates = []

for tid in sample_ids:

    print("Processing training trip:", tid)

    o, c = process_trip(
        "train",
        TRAIN_DIR / tid
    )

    sample_obs.append(o)
    sample_candidates.append(c)

sample_obs = pd.concat(sample_obs, ignore_index=True)
sample_candidates = pd.concat(
    sample_candidates, ignore_index=True
)

# Calculate candidate coverage
sample_join = sample_candidates.merge(
    sample_obs,
    on=KEY,
    how="left"
)

sample_hits = (
    sample_join[
        sample_join["edge_id"] == sample_join["ref_edge_id"]
    ][KEY]
    .drop_duplicates()
)

sample_check = sample_obs.merge(
    sample_hits.assign(covered=1),
    on=KEY,
    how="left"
)

sample_check["covered"] = (
    sample_check["covered"].fillna(0)
)

print("\nSAMPLE RESULTS")
print("Observations:", len(sample_obs))
print("Candidate rows:", len(sample_candidates))

print(
    "Candidate miss rate:",
    1 - sample_check["covered"].mean()
)

print("\nCoverage by trip:")
print(
    sample_check.groupby("trip_id")["covered"].mean()
)

Processing training trip: 00000001
Processing training trip: 00000003

SAMPLE RESULTS
Observations: 2136
Candidate rows: 14991
Candidate miss rate: 0.004213483146067398

Coverage by trip:
trip_id
00000001    0.998479
00000003    0.991474
Name: covered, dtype: float64


In [7]:
# Process all 80 training trips and 10 student-test trips

all_obs = []
all_candidates = []

for split, trip_dirs in [
    ("train", train_trips),
    ("student", test_trips)
]:

    print(f"\nStarting {split}: {len(trip_dirs)} trips")

    for i, trip_dir in enumerate(trip_dirs, 1):

        o, c = process_trip(split, trip_dir)

        all_obs.append(o)
        all_candidates.append(c)

        if i % 10 == 0 or i == len(trip_dirs):
            print(
                f"{split}: {i}/{len(trip_dirs)} completed"
            )

# Combine all trips
obs = pd.concat(all_obs, ignore_index=True)

candidates = pd.concat(
    all_candidates, ignore_index=True
)

# Validate observations
assert not obs.duplicated(KEY).any()
assert obs[KEY + ["ref_edge_id", "split"]].notna().all().all()

assert obs.groupby("split")["trip_id"].nunique().to_dict() == {
    "train": 80,
    "student": 10
}

# Validate candidates
assert candidates[KEY + ["edge_id"]].notna().all().all()

assert not candidates.duplicated(
    KEY + ["edge_id"]
).any()

# Calculate coverage
joined = candidates.merge(
    obs[KEY + ["ref_edge_id", "split"]],
    on=KEY,
    how="left",
    validate="many_to_one"
)

assert joined["ref_edge_id"].notna().all()

hit_keys = (
    joined[
        joined["edge_id"] == joined["ref_edge_id"]
    ][KEY]
    .drop_duplicates()
    .assign(covered=1)
)

check = obs.merge(
    hit_keys,
    on=KEY,
    how="left"
)

check["covered"] = check["covered"].fillna(0)

print("\n========== FINAL DATA REPORT ==========")

print("\nObservation counts:")
print(
    obs.groupby("split").agg(
        trips=("trip_id", "nunique"),
        observations=("obs_id", "size")
    )
)

print("\nCandidate counts:")
print(
    candidates.groupby(
        candidates["trip_id"].map(
            obs.drop_duplicates(KEY).set_index(KEY)["split"]
            if False else lambda x: x
        )
    ).size()
)

print("\nCandidate miss rate:")
print(1 - check.groupby("split")["covered"].mean())

# Export both files
obs.to_csv(ROOT / "observations.csv", index=False)

candidates.to_csv(
    ROOT / "candidates.csv",
    index=False
)

print("\nCSV FILES CREATED SUCCESSFULLY!")

print(ROOT / "observations.csv")
print(ROOT / "candidates.csv")


Starting train: 80 trips
train: 10/80 completed
train: 20/80 completed
train: 30/80 completed
train: 40/80 completed
train: 50/80 completed
train: 60/80 completed
train: 70/80 completed
train: 80/80 completed

Starting student: 10 trips
student: 10/10 completed

========== FINAL DATA REPORT ==========

Observation counts:
         trips  observations
split                       
student     10         19160
train       80        208417

Candidate counts:
trip_id
00000000     9902
00000001     8698
00000002    46310
00000003     6293
00000004     5851
            ...  
00000095    11553
00000096    15819
00000097    10075
00000098    27540
00000099    17049
Length: 90, dtype: int64

Candidate miss rate:
split
student    0.002923
train      0.115398
Name: covered, dtype: float64

CSV FILES CREATED SUCCESSFULLY!
/content/drive/MyDrive/PathSnap/observations.csv
/content/drive/MyDrive/PathSnap/candidates.csv


In [8]:
import pandas as pd
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/PathSnap")

obs = pd.read_csv(
    ROOT / "observations.csv",
    dtype=str
)

candidates = pd.read_csv(
    ROOT / "candidates.csv",
    dtype={"trip_id": str, "obs_id": str, "edge_id": str}
)

print("OBSERVATIONS")
print(obs.head())
print("\nShape:", obs.shape)

print("\nCANDIDATES")
print(candidates.head())
print("\nShape:", candidates.shape)

print("\nObservation columns:", obs.columns.tolist())
print("Candidate columns:", candidates.columns.tolist())

# Validate required columns
assert set([
    "trip_id", "obs_id", "ref_edge_id", "split"
]).issubset(obs.columns)

assert set([
    "trip_id", "obs_id", "edge_id",
    "distance_m", "distance_rank"
]).issubset(candidates.columns)

# Validate trip counts
print("\nTrip counts:")
print(obs.groupby("split")["trip_id"].nunique())

# Validate candidate coverage
KEY = ["trip_id", "obs_id"]

joined = candidates.merge(
    obs[KEY + ["ref_edge_id", "split"]],
    on=KEY,
    how="left",
    validate="many_to_one"
)

assert joined["ref_edge_id"].notna().all()

hits = joined[
    joined["edge_id"] == joined["ref_edge_id"]
][KEY].drop_duplicates()

coverage = obs[KEY + ["split"]].merge(
    hits.assign(covered=1),
    on=KEY,
    how="left"
)

coverage["covered"] = coverage["covered"].fillna(0)

print("\nCandidate coverage:")
print(coverage.groupby("split")["covered"].mean())

print("\nCandidate miss rate:")
print(1 - coverage.groupby("split")["covered"].mean())

print("\nDATA VALIDATION COMPLETED")

OBSERVATIONS
    trip_id obs_id ref_edge_id  split
0  00000001      1       64973  train
1  00000001      2       64973  train
2  00000001      3       64973  train
3  00000001      4       64973  train
4  00000001      5       64973  train

Shape: (227577, 4)

CANDIDATES
    trip_id obs_id edge_id  distance_m  distance_rank
0  00000001      1   64973    0.722768              1
1  00000001      1  117639   15.132321              2
2  00000001      1    2641   32.198445              3
3  00000001     10  139208    5.812021              1
4  00000001     10  133950    8.442955              2

Shape: (1198560, 5)

Observation columns: ['trip_id', 'obs_id', 'ref_edge_id', 'split']
Candidate columns: ['trip_id', 'obs_id', 'edge_id', 'distance_m', 'distance_rank']

Trip counts:
split
student    10
train      80
Name: trip_id, dtype: int64

Candidate coverage:
split
student    0.997077
train      0.884602
Name: covered, dtype: float64

Candidate miss rate:
split
student    0.002923
train     

In [9]:

from pathlib import Path
from time import perf_counter

import pandas as pd

from google.colab import drive
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    precision_recall_fscore_support
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Connect Google Drive
drive.mount("/content/drive")

# 2. Configuration
ROOT = Path("/content/drive/MyDrive/PathSnap")

RUN = ["lr"]

# IMPORTANT: Only the two features present in your CSV
FEATURES = ["distance_m", "distance_rank"]

OUT = ROOT / "baseline_results"
OUT.mkdir(exist_ok=True)

KEY = ["trip_id", "obs_id"]

# 3. Load shared CSV files
obs = pd.read_csv(
    ROOT / "observations.csv",
    dtype=str
)

pairs = pd.read_csv(
    ROOT / "candidates.csv",
    usecols=KEY + ["edge_id"] + FEATURES,
    dtype={
        "trip_id": str,
        "obs_id": str,
        "edge_id": str
    }
)

# 4. Verify official splits
train_obs = obs[obs["split"].eq("train")]
student_obs = obs[obs["split"].eq("student")]

assert train_obs["trip_id"].nunique() == 80
assert student_obs["trip_id"].nunique() == 10

assert set(train_obs["trip_id"]).isdisjoint(
    student_obs["trip_id"]
)

assert not obs.duplicated(KEY).any()

# 5. Create binary classification labels
rows = pairs.merge(
    obs,
    on=KEY,
    how="left",
    validate="many_to_one"
)

assert rows["ref_edge_id"].notna().all()

rows["label"] = (
    rows["edge_id"] == rows["ref_edge_id"]
).astype(int)

train = rows[rows["split"].eq("train")]
student = rows[rows["split"].eq("student")]

# 6. Three-fold cross-validation grouped by trip
folds = list(
    GroupKFold(n_splits=3).split(
        train_obs,
        groups=train_obs["trip_id"]
    )
)

# 7. Logistic Regression
models = {
    "lr": LogisticRegression(max_iter=150)
}


def fit_and_score(name, fit_rows, test_rows, truth):

    steps = [
        SimpleImputer(
            strategy="median",
            add_indicator=True
        ),
        StandardScaler(),
        clone(models[name])
    ]

    model = make_pipeline(*steps)

    start = perf_counter()

    model.fit(
        fit_rows[FEATURES],
        fit_rows["label"]
    )

    fit_s = perf_counter() - start

    start = perf_counter()

    prob = model.predict_proba(
        test_rows[FEATURES]
    )[:, 1]

    predict_s = perf_counter() - start

    y = test_rows["label"]
    hard = prob >= 0.5

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y,
            hard,
            average="binary",
            zero_division=0
        )
    )

    tn, fp, fn, tp = confusion_matrix(
        y,
        hard,
        labels=[0, 1]
    ).ravel()

    scored = test_rows[
        KEY + ["edge_id", "label"]
    ].assign(score=prob)

    best = (
        scored.sort_values(
            "score",
            ascending=False,
            kind="stable"
        )
        .drop_duplicates(KEY)
    )

    point = truth[KEY].merge(
        best[KEY + ["label"]],
        on=KEY,
        how="left"
    )

    point["correct"] = point["label"].fillna(0)

    covered = scored.groupby(KEY)["label"].max()

    coverage = truth[KEY].join(
        covered,
        on=KEY
    )["label"].fillna(0)

    metrics = dict(
        row_accuracy=accuracy_score(y, hard),
        precision=precision,
        recall=recall,
        f1=f1,
        log_loss=log_loss(y, prob, labels=[0, 1]),
        tn=tn,
        fp=fp,
        fn=fn,
        tp=tp,
        top1_trip_mean=(
            point.groupby("trip_id")["correct"]
            .mean()
            .mean()
        ),
        top1_pooled=point["correct"].mean(),
        candidate_miss=1 - coverage.mean(),
        fit_s=fit_s,
        predict_s=predict_s
    )

    return metrics, scored.drop(columns="label")


# 8. Run three grouped CV folds
for name in RUN:

    results = []

    for fold, (fit_index, val_index) in enumerate(
        folds,
        start=1
    ):

        fit_trips = train_obs.iloc[
            fit_index
        ]["trip_id"]

        val_obs = train_obs.iloc[val_index]

        fit_rows = train[
            train["trip_id"].isin(fit_trips)
        ]

        val_rows = train[
            train["trip_id"].isin(val_obs["trip_id"])
        ]

        metrics, _ = fit_and_score(
            name,
            fit_rows,
            val_rows,
            val_obs
        )

        results.append(
            dict(
                model=name,
                split="cv",
                fold=fold,
                **metrics
            )
        )

        elapsed = metrics["fit_s"] + metrics["predict_s"]

        print(
            f"{name} fold {fold}: "
            f"{elapsed:.1f}s, "
            f"top-1={metrics['top1_trip_mean']:.3f}",
            flush=True
        )

    # 9. Train on all 80 trips and evaluate 10 student trips
    metrics, scores = fit_and_score(
        name,
        train,
        student,
        student_obs
    )

    results.append(
        dict(
            model=name,
            split="student",
            **metrics
        )
    )

    # 10. Save results
    pd.DataFrame(results).to_csv(
        OUT / f"{name}_metrics.csv",
        index=False
    )

    scores.to_csv(
        OUT / f"{name}_student_scores.csv",
        index=False
    )

    print(
        f"\n{name} COMPLETE! "
        f"Student top-1={metrics['top1_trip_mean']:.4f}",
        flush=True
    )

print("\nALL LOGISTIC REGRESSION RUNS COMPLETED!")
print("Results folder:", OUT)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
lr fold 1: 1.8s, top-1=0.577
lr fold 2: 1.6s, top-1=0.614
lr fold 3: 0.9s, top-1=0.580

lr COMPLETE! Student top-1=0.5781

ALL LOGISTIC REGRESSION RUNS COMPLETED!
Results folder: /content/drive/MyDrive/PathSnap/baseline_results


In [10]:
import pandas as pd

OUT = ROOT / "baseline_results"

metrics = pd.read_csv(
    OUT / "lr_metrics.csv"
)

scores = pd.read_csv(
    OUT / "lr_student_scores.csv"
)

print("LOGISTIC REGRESSION METRICS")
display(metrics)

print("\nSTUDENT PREDICTION FILE")
display(scores.head())

print("\nVerification:")
print("CV folds:", len(metrics[metrics["split"] == "cv"]))
print("Student rows:", len(metrics[metrics["split"] == "student"]))
print("Student candidate scores:", len(scores))

assert len(metrics[metrics["split"] == "cv"]) == 3
assert len(metrics[metrics["split"] == "student"]) == 1
assert len(scores) > 0

print("\nLOGISTIC REGRESSION VERIFICATION PASSED!")

LOGISTIC REGRESSION METRICS


,model,split,fold,row_accuracy,precision,recall,f1,log_loss,tn,fp,fn,tp,top1_trip_mean,top1_pooled,candidate_miss,fit_s,predict_s
0,lr,cv,1.0,0.862858,0.585574,0.532616,0.557841,0.271210,243113,19173,23773,27091,0.577024,0.432249,0.269279,1.734426,0.049608
1,lr,cv,2.0,0.856741,0.606525,0.537651,0.570015,0.284300,294417,23808,31559,36699,0.614427,0.595535,0.014425,1.516727,0.100711
2,lr,cv,3.0,0.843480,0.538726,0.526654,0.532621,0.287881,290614,29421,30883,34361,0.580169,0.532350,0.061939,0.838712,0.053509
3,lr,student,NaN,0.854058,0.574647,0.507328,0.538894,0.279741,87370,7174,9412,9692,0.578103,0.557620,0.002923,1.242665,0.035132



STUDENT PREDICTION FILE


,trip_id,obs_id,edge_id,score
0,0,1,17630,0.579553
1,0,1,17634,0.313490
2,0,1,17635,0.043028
3,0,1,17668,0.014676
4,0,1,17633,0.001756



Verification:
CV folds: 3
Student rows: 1
Student candidate scores: 113648

LOGISTIC REGRESSION VERIFICATION PASSED!


In [11]:
from pathlib import Path
import pandas as pd

OUT = Path("/content/drive/MyDrive/PathSnap/baseline_results")

print("Files generated:")
for f in sorted(OUT.glob("*")):
    print(f.name)

metrics = pd.read_csv(OUT / "lr_metrics.csv")

print("\nLogistic Regression results:")
display(metrics)

assert len(metrics[metrics["split"] == "cv"]) == 3
assert len(metrics[metrics["split"] == "student"]) == 1
assert (OUT / "lr_student_scores.csv").exists()

print("\nPERSON 1 LOGISTIC REGRESSION CHECK PASSED")

Files generated:
lr_metrics.csv
lr_student_scores.csv

Logistic Regression results:


,model,split,fold,row_accuracy,precision,recall,f1,log_loss,tn,fp,fn,tp,top1_trip_mean,top1_pooled,candidate_miss,fit_s,predict_s
0,lr,cv,1.0,0.862858,0.585574,0.532616,0.557841,0.271210,243113,19173,23773,27091,0.577024,0.432249,0.269279,1.734426,0.049608
1,lr,cv,2.0,0.856741,0.606525,0.537651,0.570015,0.284300,294417,23808,31559,36699,0.614427,0.595535,0.014425,1.516727,0.100711
2,lr,cv,3.0,0.843480,0.538726,0.526654,0.532621,0.287881,290614,29421,30883,34361,0.580169,0.532350,0.061939,0.838712,0.053509
3,lr,student,NaN,0.854058,0.574647,0.507328,0.538894,0.279741,87370,7174,9412,9692,0.578103,0.557620,0.002923,1.242665,0.035132



PERSON 1 LOGISTIC REGRESSION CHECK PASSED
